In [ ]:
# PUBLIC_MATERIALS_NOTEBOOK_SETUP
from pathlib import Path
from urllib.request import urlretrieve

try:
    import google.colab  # type: ignore  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

_MATERIALS_BASE = \"https://raw.githubusercontent.com/yonpak/ai-from-zero-to-builder-materials/main/labs/notebooks/tiny-llm/\"
if _IN_COLAB or not Path(\"tiny_llm.py\").exists():
    for _name in (\"tiny_llm.py\", \"tiny_corpus.txt\"):
        urlretrieve(_MATERIALS_BASE + _name, _name)


# Assemble, Train, and Sample a Tiny Decoder-Only LM

This notebook supports `l05-13-putting-the-pieces-together`, `l06-05-assemble-the-model`,
`l06-06-training-loop`, and `l06-09-sampling-text`.

The default run uses CPU. It is intentionally tiny so GPU access is optional.

In [ ]:
from pathlib import Path
import torch

from tiny_llm import (
    SimpleTokenizer,
    TinyConfig,
    TinyDecoderLM,
    count_parameters,
    set_seed,
    train_steps,
)

SEED = 7
set_seed(SEED)

text = Path("tiny_corpus.txt").read_text(encoding="utf-8")
tokenizer = SimpleTokenizer.from_text(text)
token_ids = tokenizer.encode(text)

config = TinyConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=12,
    n_embd=48,
    n_head=4,
    n_layer=2,
    dropout=0.0,
)
model = TinyDecoderLM(config)

print(config)
print("parameters:", count_parameters(model))

In [ ]:
x = torch.tensor([token_ids[:config.block_size]], dtype=torch.long)
y = torch.tensor([token_ids[1:config.block_size + 1]], dtype=torch.long)
logits, initial_batch_loss = model(x, y)

print("logits:", tuple(logits.shape))
print("initial batch loss:", float(initial_batch_loss.detach()))
assert logits.shape == (1, config.block_size, tokenizer.vocab_size)
assert torch.isfinite(initial_batch_loss)

## Train

For the teaching run, `80` CPU steps are enough to see the loss move downward on the repeated fixture.
The exact decimal value is not a golden result.

In [ ]:
losses = train_steps(
    model,
    token_ids,
    steps=80,
    batch_size=16,
    learning_rate=3e-3,
    seed=SEED,
    device="cpu",
)

print("first loss:", losses[0])
print("best of final 10:", min(losses[-10:]))
assert min(losses[-10:]) < losses[0]

## Sample

Temperature rescales logits before softmax. `top_k` keeps only the most likely few choices.
The explicit sampling seed makes repeated generation deterministic for the same trained model.

In [ ]:
prompt = torch.tensor([tokenizer.encode("the moon")], dtype=torch.long)

sample_a = model.generate(
    prompt,
    max_new_tokens=24,
    temperature=0.8,
    top_k=8,
    seed=11,
)
sample_b = model.generate(
    prompt,
    max_new_tokens=24,
    temperature=0.8,
    top_k=8,
    seed=11,
)

assert torch.equal(sample_a, sample_b)
print(tokenizer.decode(sample_a[0].tolist()))

## Optional checkpoint

If you want a local checkpoint, save it after training:

```python
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": config.as_dict(),
        "vocab": tokenizer.itos,
        "seed": SEED,
    },
    "tiny_llm_checkpoint.pt",
)
```

The repository does not commit a generated checkpoint. Re-running training is the reproducible source.

## Failure checks

- If `n_embd` is not divisible by `n_head`, model construction must raise an error.
- If the sequence is longer than `block_size`, the model must raise an error.
- If temperature is `0` or negative, sampling must raise an error.
- If loss becomes `NaN`, reduce the learning rate and verify batch/target shapes.

See `EXPECTED_OUTPUTS.md` and `REPRODUCIBILITY.md` for the shared validation contract.